walking cost raster creation

In [4]:
import os

proj_dir = "/opt/miniconda3/envs/KGG375_AT3/share/proj"
os.environ["PROJ_LIB"] = proj_dir
os.environ["PROJ_DATA"] = proj_dir

import pyproj
pyproj.datadir.set_data_dir(proj_dir)

import numpy as np
from whitebox.whitebox_tools import WhiteboxTools
import heapq
import rasterio
import geopandas as gpd

# Quick verification test (will print and ensure it's unblocked)
print("PROJ active target directory:", pyproj.datadir.get_data_dir())
print("EPSG:7855 resolves:", pyproj.CRS.from_user_input("EPSG:7855"))


PROJ active target directory: /opt/miniconda3/envs/KGG375_AT3/share/proj
EPSG:7855 resolves: EPSG:7855


In [5]:
#intialise whitebox tools
wbt = WhiteboxTools()
wbt.verbose = False

In [6]:

dem = "/Users/sandencook/Documents/AT3_Group6_2026/data/processed/dem_10m.tif"
slope_raster = "/Users/sandencook/Documents/AT3_Group6_2026/data/processed/slope_deg.tif"
aspect_raster = "/Users/sandencook/Documents/AT3_Group6_2026/data/processed/aspect.tif"
suitability_raster = "/Users/sandencook/Documents/AT3_Group6_2026/data/processed/suitability_wlc.tif"

In [4]:
#calculate slope and aspect rasters
wbt.slope(dem=dem, output=slope_raster, zfactor=1.0)
wbt.aspect(dem=dem, output=aspect_raster)

0

In [28]:

#read slope and aspect rasters

with rasterio.open(slope_raster) as src:
    slope = src.read(1)
    meta = src.meta
with rasterio.open(aspect_raster) as src:
    aspect = src.read(1)

with rasterio.open(suitability_raster) as src:
    suitability = src.read(1)


In [29]:
#no data handling
slope = np.where(slope == meta['nodata'], 0, slope)
aspect = np.where(aspect == meta['nodata'], 0, aspect)
suitability = np.where(suitability == meta['nodata'], 0, suitability)

In [7]:

DIRECTIONS = {
    (-1, 0): 0,    # North
    (-1, 1): 45,   # Northeast
    (0, 1): 90,    # East
    (1, 1): 135,   # Southeast
    (1, 0): 180,   # South
    (1, -1): 225,  # Southwest
    (0, -1): 270,  # West
    (-1, -1): 315  # Northwest
}

In [8]:
def calculate_directional_cost(slope_deg, aspect_deg, move_dir):
    """
    Compute cost based on how closely movement matches the slope's aspect.
    """
    move_angle = DIRECTIONS[move_dir]
        # Angular difference between terrain aspect (downhill direction) and travel direction
    
    angle_diff = np.abs(aspect_deg - move_angle)
    if angle_diff > 180:
        angle_diff = 360 - angle_diff
    # angle_diff near 0 means moving downhill
    # angle_diff near 180 means moving directly uphill
    # Scale slope penalty based on direction alignment
    directional_modifier = np.cos(np.radians(angle_diff)) # -1 (uphill) to 1 (downhill)
    
    # Simple Hiking-Inspired Cost Model (penalty for steep climbs)
    # If moving uphill (modifier is negative), increase base cost heavily by slope factor
    if directional_modifier < 0: 
        cost = 1.0 + (slope_deg * 0.1 * np.abs(directional_modifier))
    else: # Moving downhill or cross-slope
        cost = 1.0 + (slope_deg * 0.02 * (1 - directional_modifier)) 
    return max(0.1, cost)  # Prevent negative/zero weights

In [ ]:
#test

#test = suitability_raster[1, 1]
#print(f"{test}")


#test

test = suitability[1, 1]
print(f"{test}")

test2 = slope[1, 1]
print(f"{test}")

test2 = slope[1, 1]
print(f"{test}")


0.5042000412940979


In [9]:
#Dijkstra for Minimum Cost Path
def anisotropic_least_cost_path(slope, aspect, suitability, start, end):
    rows, cols = slope.shape
    # Priority Queue: (cumulative_cost, current_row, current_col, path_history)
    queue = [(0.0, start[0], start[1], [start])]
    visited = set()
    
    while queue:
        curr_cost, r, c, path = heapq.heappop(queue)
        
        if (r, c) == end:
            return path, curr_cost
            
        if (r, c) in visited:
            continue
        visited.add((r, c))
        
        for move_dir in DIRECTIONS.keys():
            nr, nc = r + move_dir[0], c + move_dir[1]
            
            if 0 <= nr < rows and 0 <= nc < cols:
                if (nr, nc) in visited:
                    continue
                
                # Diagonal distance adjustment (multiplied by sqrt(2) if diagonal)
                dist_factor = 1.414214 if (move_dir[0] != 0 and move_dir[1] != 0) else 1.0
                
                # Fetch target cell terrain metadata
                cell_slope = slope[nr, nc]
                cell_aspect = aspect[nr, nc]
                suitability_value = suitability[nr, nc]
                
                # Evaluate anisotropic edge weight
                edge_cost = calculate_directional_cost(cell_slope, cell_aspect, move_dir) * dist_factor
                
                totalcost = edge_cost * suitability_value
                
                heapq.heappush(queue, (curr_cost + totalcost, nr, nc, path + [(nr, nc)]))
                
    return None, float('inf')

In [10]:
import os
import pyproj

proj_dir = "/opt/miniconda3/envs/KGG375_AT3/share/proj"
if not os.path.exists(proj_dir):
    raise FileNotFoundError(f"PROJ data directory not found: {proj_dir}")

os.environ["PROJ_LIB"] = proj_dir
os.environ["PROJ_DATA"] = proj_dir
pyproj.datadir.set_data_dir(proj_dir)

start_shapefile = "/Users/sandencook/Documents/AT3_Group6_2026/data/raw/points/startend_list.shp"
end_shapefile = "/Users/sandencook/Documents/AT3_Group6_2026/data/raw/points/summit_list.shp"

# Ensure the PROJ database is configured before reading shapefiles with CRS EPSG:7855
gdf_start = gpd.read_file(start_shapefile)
gdf_end = gpd.read_file(end_shapefile)

with rasterio.open(slope_raster) as src:
    raster_crs = src.crs
    
    if gdf_start.crs != raster_crs:
        gdf_start = gdf_start.to_crs(raster_crs)
    if gdf_end.crs != raster_crs:
        gdf_end = gdf_end.to_crs(raster_crs)

    #extract first x and y coordinates from the shapefiles
    start_geom = gdf_start.geometry.iloc[0]
    end_geom = gdf_end.geometry.iloc[0]
    
    #convert to row indices in the raster
    start_node = src.index(start_geom.x, start_geom.y)
    end_node = src.index(end_geom.x, end_geom.y)
    
print(f"Shapefile coordinate coordinates mapped to array space:")
print(f"  Start Node (Row, Col): {start_node}")
print(f"  End Node (Row, Col): {end_node}")


Shapefile coordinate coordinates mapped to array space:
  Start Node (Row, Col): (479, 100)
  End Node (Row, Col): (642, 430)


In [ ]:
optimal_path, total_cost = anisotropic_least_cost_path(slope, aspect, suitability, start_node, end_node)
print(f"Minimum cost path computed. Total Directional Cost: {total_cost:.2f}")

In [ ]:
#export the path to raster

# Create a blank array of zeros matching terrain dimensions
path_mask = np.zeros_like(slope, dtype=np.uint8)

# Populate the cells along the optimal path with a value of 1
if optimal_path:
    for row, col in optimal_path:
        path_mask[row, col] = 1

# Write the Raster Mask to Disk using your original spatial metadata
output_raster = f"minimum_cost_path{total_cost:.0f}.tif"

# Update metadata for a single-band, 8-bit unsigned integer raster
meta.update(dtype=rasterio.uint8, count=1, nodata=0, compress='lzw')

with rasterio.open(output_raster, "w", **meta) as dst:
    dst.write(path_mask, 1)

print(f"Success! Minimum cost path raster exported to: {output_raster}")

Success! Minimum cost path raster exported to: minimum_cost_path889.tif


In [ ]:
import pyproj
print(pyproj.datadir.get_data_dir())

/opt/miniconda3/envs/KGG375_AT3/share/proj
